# Semantic Integrity Model — Training and Validation

This notebook runs the measured training, validation, packaging, and deployment pipeline for the semantic-integrity classifier.

It preserves canonical Arabic and trusted references, prevents ayah-level leakage, calibrates thresholds on validation data only, evaluates a frozen held-out split, packages the checkpoint, and validates the inference contract.


In [ ]:
# 1. Initialize the project workspace
from pathlib import Path
from urllib.request import urlopen
import json, os, platform, shutil, subprocess, sys

REPOSITORY_ID = 1391259908
with urlopen(f"https://api.github.com/repositories/{REPOSITORY_ID}", timeout=30) as response:
    repository_metadata = json.load(response)

repo_url = repository_metadata["clone_url"]
workspace = Path("/content/semantic_integrity_project")

if workspace.exists():
    shutil.rmtree(workspace)

subprocess.run(["git", "clone", "--depth", "1", repo_url, str(workspace)], check=True)
os.chdir(workspace)

print("Repository resolved:", repository_metadata["full_name"])
print("Workspace:", workspace)
print("Python:", platform.python_version())


In [ ]:
# 2. Prepare the Colab environment and run deterministic quality gates
import importlib.util
import time

packages = [
    "pydantic>=2.8,<3",
    "fastapi>=0.115,<1",
    "scikit-learn>=1.5",
    "transformers>=4.45",
    "pytest>=8",
    "httpx>=0.27",
    "requests>=2.32",
    "huggingface_hub>=0.35"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True
)

last_error = None
for attempt in range(1, 4):
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", *packages],
            check=True
        )
        last_error = None
        break
    except subprocess.CalledProcessError as exc:
        last_error = exc
        if attempt < 3:
            print(f"Dependency installation retry {attempt}/3...")
            time.sleep(5 * attempt)

if last_error is not None:
    raise last_error

subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "-m", "training.train", "--smoke"], check=True)
subprocess.run(
    [sys.executable, "-m", "compileall", "-q",
     "amanah_engine", "api", "data", "training", "scripts", "deployment"],
    check=True
)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU runtime is required for measured fine-tuning."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 3. Verify trusted sources and retrieve the reference bundle
subprocess.run([sys.executable, "scripts/source_contract_smoke.py", "--timeout", "90"], check=True)

Path("data/private").mkdir(parents=True, exist_ok=True)
Path("artifacts/semantic_dataset").mkdir(parents=True, exist_ok=True)
Path("checkpoints").mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "scripts/fetch_verified_sources.py",
    "--output", "data/private/reference_bundle.json",
    "--runtime-store", "data/reference_store.json"
], check=True)

with open("data/private/reference_bundle.json", encoding="utf-8") as handle:
    bundle = json.load(handle)

assert len(bundle["ayahs"]) == 6236
assert all(len(row["references"]) == 3 for row in bundle["ayahs"])
print("Verified canonical ayat:", len(bundle["ayahs"]))


In [ ]:
# 4. Build the training dataset and enforce dataset QA
subprocess.run([
    sys.executable, "-m", "scripts.prepare_amanah_sd",
    "data/private/reference_bundle.json",
    "artifacts/semantic_dataset",
    "--seed", "42"
], check=True)

subprocess.run([
    sys.executable, "scripts/qa_dataset.py",
    "--reference-bundle", "data/private/reference_bundle.json",
    "--split-dir", "artifacts/semantic_dataset",
    "--output", "artifacts/semantic_dataset/qa_report.json"
], check=True)

with open("artifacts/semantic_dataset/qa_report.json", encoding="utf-8") as handle:
    qa = json.load(handle)

assert qa["status"] == "ok"
assert qa["split_ayah_leakage"] == []
print(json.dumps(qa, ensure_ascii=False, indent=2))


In [ ]:
# 5. Fine-tune the semantic-drift classifier
subprocess.run([
    sys.executable, "-m", "training.train",
    "--train", "artifacts/semantic_dataset/train.jsonl",
    "--validation", "artifacts/semantic_dataset/validation.jsonl",
    "--output-dir", "checkpoints/semantic-integrity-v0.1",
    "--epochs", "5",
    "--batch-size", "4",
    "--gradient-accumulation", "4",
    "--learning-rate", "2e-5",
    "--max-length", "256",
    "--patience", "2"
], check=True)


In [ ]:
# 6. Calibrate thresholds and evaluate the frozen held-out test split
subprocess.run([
    sys.executable, "-m", "training.calibrate_thresholds",
    "--checkpoint", "checkpoints/semantic-integrity-v0.1",
    "--validation", "artifacts/semantic_dataset/validation.jsonl"
], check=True)

subprocess.run([
    sys.executable, "-m", "training.evaluate",
    "--checkpoint", "checkpoints/semantic-integrity-v0.1",
    "--test", "artifacts/semantic_dataset/test.jsonl",
    "--output", "artifacts/evaluation"
], check=True)

with open("artifacts/evaluation/metrics.json", encoding="utf-8") as handle:
    metrics = json.load(handle)

assert metrics.get("status") == "ok"
for key in ["macro_f1", "critical_drift_recall", "false_safe_rate", "severity_macro_f1"]:
    assert 0.0 <= metrics[key] <= 1.0

print(json.dumps(metrics, indent=2))


In [ ]:
# 7. Package the model and validate the inference handler locally
subprocess.run([
    sys.executable, "scripts/package_hf_model.py",
    "--checkpoint", "checkpoints/semantic-integrity-v0.1",
    "--reference-store", "data/reference_store.json",
    "--output", "artifacts/model_repository"
], check=True)

required_files = [
    "handler.py",
    "model.pt",
    "amanah_config.json",
    "thresholds.json",
    "reference_store.json",
    "requirements.txt"
]
missing = [name for name in required_files if not (Path("artifacts/model_repository") / name).exists()]
assert not missing, f"Missing deployment files: {missing}"

sys.path.insert(0, str(Path("artifacts/model_repository").resolve()))
from handler import EndpointHandler

handler = EndpointHandler("artifacts/model_repository")
with open("artifacts/model_repository/reference_store.json", encoding="utf-8") as handle:
    reference_store = json.load(handle)

row = reference_store["ayahs"][0]
response = handler({"inputs": {
    "source_type": "quran",
    "source_ar": row["source_ar"],
    "candidate_en": row["references_en"][0],
    "ayah_id": row["ayah_id"]
}})

assert response["decision"] in {"PASS", "REVIEW", "CRITICAL", "ABSTAIN"}
assert response["reference_status"] == "verified"
print(json.dumps(response, ensure_ascii=False, indent=2))


In [ ]:
# 8. Write the measured validation report and release archive
report = {
    "dataset_summary": json.load(open("artifacts/semantic_dataset/dataset_summary.json", encoding="utf-8")),
    "dataset_qa": qa,
    "evaluation": metrics,
    "model_repository_url": None,
    "inference_endpoint_url": None,
    "deployment_status": "not_started"
}

Path("artifacts/VALIDATION_REPORT.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

shutil.make_archive(
    "/content/SEMANTIC_INTEGRITY_MODEL_RELEASE",
    "zip",
    "artifacts/model_repository"
)

print(json.dumps(report, ensure_ascii=False, indent=2))
print("Release archive:", "/content/SEMANTIC_INTEGRITY_MODEL_RELEASE.zip")


In [ ]:
# 9. Publish the validated package to Hugging Face when authentication is available
from huggingface_hub import HfApi, create_repo, get_token, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    print("Hugging Face authentication is required only for publishing/deployment.")
    notebook_login()
    hf_token = get_token()

if not hf_token:
    raise RuntimeError("Hugging Face authentication was not completed.")

api = HfApi(token=hf_token)
identity = api.whoami()
namespace = identity.get("name") or identity.get("fullname")
model_repo_id = f"{namespace}/semantic-integrity-v0-1"

create_repo(
    repo_id=model_repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
    token=hf_token
)

api.upload_folder(
    repo_id=model_repo_id,
    repo_type="model",
    folder_path="artifacts/model_repository",
    token=hf_token,
    commit_message="Publish validated v0.1 checkpoint"
)

api.upload_file(
    path_or_fileobj="artifacts/VALIDATION_REPORT.json",
    path_in_repo="VALIDATION_REPORT.json",
    repo_id=model_repo_id,
    repo_type="model",
    token=hf_token,
    commit_message="Add measured validation report"
)

report["model_repository_url"] = f"https://huggingface.co/{model_repo_id}"
report["deployment_status"] = "model_uploaded"

Path("artifacts/VALIDATION_REPORT.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Model repository:", report["model_repository_url"])
print("Training and validation are complete.")
print("Endpoint deployment can now use the packaged custom handler.")
